In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

# 选择一个嵌入模型，这里以 'sentence-transformers/all-MiniLM-L6-v2' 为例
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def embed_text(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs, return_dict=True).pooler_output
    return embeddings.cpu().numpy()


In [1]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection

# 连接到 Milvus 服务
connections.connect("default", host="localhost", port="19530")

# 定义集合的 schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # dim需要和嵌入的维度匹配
    FieldSchema(name="knowledge", dtype=DataType.VARCHAR, max_length=1024),
    # FieldSchema(name="matchingknowledge", dtype=DataType.VARCHAR, max_length=1024),
    # FieldSchema(name="topic", dtype=DataType.VARCHAR, max_length=128),
    FieldSchema(name="flag", dtype=DataType.INT64),
    FieldSchema(name="confidence_flag", dtype=DataType.FLOAT),
    # FieldSchema(name="contradiction_flag", dtype=DataType.FLOAT),
    FieldSchema(name="score_of_confidence", dtype=DataType.FLOAT),
    # FieldSchema(name="score_of_contradiction", dtype=DataType.FLOAT)
]
schema = CollectionSchema(fields,description="Biological", auto_id=True)

# 创建集合
collection = Collection("biological_qwen_basic_special", schema)


In [1]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection

# 连接到 Milvus 服务
connections.connect("default", host="localhost", port="19530")

# 定义集合的 schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # dim需要和嵌入的维度匹配
    FieldSchema(name="knowledge", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="topic", dtype=DataType.VARCHAR, max_length=128),
    FieldSchema(name="flag", dtype=DataType.INT64),
    FieldSchema(name="Confidence", dtype=DataType.FLOAT)
]
schema = CollectionSchema(fields,description="Iterative dataset which contains all data should be detect", auto_id=True)

# 创建集合
collection = Collection("basic_exp5_qwen2Finetun", schema)

In [2]:
from pymilvus import utility
# 创建索引
index_params = {"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 128}}
collection.create_index("embedding", index_params)

Status(code=0, message=)

In [24]:
from datasets import load_dataset

# 加载数据集
dataset = load_dataset("allenai/ai2_arc",'ARC-Easy')

# 查看数据集的一部分
print(dataset['train'])


Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 2251
})


In [10]:
def assemble_question_answers_with_flags(data):
    """
    根据输入字典组装问题和所有答案，并为每个答案添加一个标识符，正确为1，错误为0。

    参数:
    data (dict): 包含问题、选项和正确答案的字典。

    返回:
    list: 包含组装好的问题和答案字符串的字典，每个字典中包含答案和标识符。
    """
    # 提取问题
    question = data['question']

    # 获取正确答案的标签
    answer_key = data['answerKey']

    # 获取所有选项的文本和标签
    choices = data['choices']['text']
    labels = data['choices']['label']

    # 找到正确答案的索引
    correct_index = labels.index(answer_key)

    # 生成包含所有答案及其标识符的列表
    results = []
    for i, choice in enumerate(choices):
        flag = 1 if i == correct_index else 0
        result = {
            "question": question,
            "answer": choice,
            "flag": flag
        }
        results.append(result)

    return results

# 示例数据
# 输出结果

for result in assemble_question_answers_with_flags(dataset['train'][0]):
    print(result)

{'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'answer': 'dry palms', 'flag': 1}
{'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'answer': 'wet palms', 'flag': 0}
{'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'answer': 'palms covered with oil', 'flag': 0}
{'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'answer': 'palms covered with lotion', 'flag': 0}


In [31]:
import json
import numpy as np
from tqdm import tqdm

def assemble_question_answers_with_flags(data):
    """
    根据输入字典组装问题和所有答案，并为每个答案添加一个标识符，正确为1，错误为0。

    参数:
    data (dict): 包含问题、选项和正确答案的字典。

    返回:
    list: 包含组装好的问题和答案字符串的字典，每个字典中包含答案和标识符。
    """
    question = data['question']
    answer_key = data['answerKey']
    choices = data['choices']['text']
    labels = data['choices']['label']
    correct_index = labels.index(answer_key)

    results = []
    for i, choice in enumerate(choices):
        flag = 1 if i == correct_index else 0
        result = f"{question} {choice}"
        results.append({"text": result, "flag": flag})

    return results

def embed_data_to_json(dataset, output_file):
    data_list = []

    for i, item in tqdm(enumerate(dataset['train']), total=len(dataset['train']), desc="Processing embeddings"):
        qa_pairs = assemble_question_answers_with_flags(item)  # 组装问题和答案对

        for qa in qa_pairs:
            text = qa['text']
            flag = qa['flag']
            embedding = embed_text(text)  # 获取嵌入向量

            # 如果 embedding 是 NumPy 数组，直接转换为 list
            embedding_np = embedding.astype(np.float32).tolist()

            # 构建保存的数据结构
            data_item = {
                'id': i,
                'embedding': embedding_np[0],
                'knowledge': text,
                'flag': flag
            }

            data_list.append(data_item)

    # 保存为 JSON 文件
    with open(output_file, 'w') as f:
        json.dump(data_list, f, indent=4)

# 使用该函数将数据保存为 JSON
output_file = 'embeddings-challenge.json'
embed_data_to_json(dataset, output_file)


Processing embeddings: 100%|██████████| 1119/1119 [00:31<00:00, 35.17it/s]


In [1]:
from datasets import load_dataset
import json

# 加载数据集
dataset = load_dataset("allenai/ai2_arc", "'ARC-Challenge'")

def get_incorrect_answers(data):
    question = data['question']
    correct_answer_key = data['answerKey']

    # 找到正确答案的索引
    correct_index = data['choices']['label'].index(correct_answer_key)

    # 获取所有非正确答案的文本
    incorrect_answers = [data['choices']['text'][i] for i in range(len(data['choices']['text'])) if i != correct_index]

    # 将问题和每个非正确答案组合成字符串
    result = [f"{question} {answer}" for answer in incorrect_answers]

    return result

# 处理整个数据集并将结果存储为列表
processed_data = []

for item in dataset['train']:
    incorrect_answers_str = get_incorrect_answers(item)
    processed_data.extend(incorrect_answers_str)  # 合并所有非正确答案的字符串

# 将结果保存为 JSON 文件
output_file = 'arc_easy_incorrect_answers.json'
with open(output_file, 'w') as f:
    json.dump(processed_data, f, indent=4)

print(f"Processed data saved to {output_file}")


ValueError: Config name is missing.
Please pick one among the available configs: ['ARC-Challenge', 'ARC-Easy']
Example of usage:
	`load_dataset('ai2_arc', 'ARC-Challenge')`

In [25]:
# 直接插入到数据库中
# 使用milvus wrapper

from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import numpy as np
from tqdm import tqdm
from tool.Milvus.MilvusWrapper import MilvusWrapper
import os
from dotenv import load_dotenv
load_dotenv()

def assemble_question_answers_with_flags(data):
    """
    根据输入字典组装问题和所有答案，并为每个答案添加一个标识符，正确为1，错误为0。

    参数:
    data (dict): 包含问题、选项和正确答案的字典。

    返回:
    list: 包含组装好的问题和答案字符串的字典，每个字典中包含答案和标识符。
    """
    question = data['question']
    answer_key = data['answerKey']
    choices = data['choices']['text']
    labels = data['choices']['label']
    correct_index = labels.index(answer_key)

    results = []
    for i, choice in enumerate(choices):
        flag = 1 if i == correct_index else 0
        result = f"{question} {choice}"
        results.append({"text": result, "flag": flag})

    return results

def insert_data_to_milvus(dataset, collection_name):
    # 连接到 Milvus 服务
    # connections.connect("default", host="localhost", port="19530")
    # 获取 collection
    # collection = Collection(name=collection_name)

    milvus_wrapper = MilvusWrapper(host='localhost', port='19530', collection_name=collection_name)
    for i, item in tqdm(enumerate(dataset['train']), total=len(dataset['train']), desc="Processing embeddings"):
        qa_pairs = assemble_question_answers_with_flags(item)  # 组装问题和答案对

        for qa in qa_pairs:
            text = qa['text']
            flag = qa['flag']
            embedding = embed_text(text)  # 获取嵌入向量

            # 如果 embedding 是 NumPy 数组，直接转换为 list
            embedding_np = embedding.astype(np.float32).tolist()
            milvus_wrapper.insert_data([
                embedding_np,  # embedding
                [text],  # knowledge
                [flag]])
            # 插入数据到 Milvus

# 定义你的 collection 名称
collection_name = "iterative_dataset"

# 调用函数将数据插入 Milvus
insert_data_to_milvus(dataset, collection_name)


Connected to Milvus collection: iterative_dataset
Model sentence-transformers/all-MiniLM-L6-v2 loaded and moved to cuda.


Processing embeddings: 100%|██████████| 2251/2251 [06:38<00:00,  5.65it/s]


In [4]:
import json

def separate_data_by_flag(input_file, output_file_flag_0, output_file_flag_1):
    # 读取输入的 JSON 文件
    with open(input_file, 'r') as f:
        data = json.load(f)

    # 初始化两个列表用于存储 flag 为 0 和 flag 为 1 的数据
    flag_0_data = []
    flag_1_data = []

    # 遍历数据，按照 flag 分离
    for item in data:
        if item.get('flag') == 0:
            flag_0_data.append(item)
        elif item.get('flag') == 1:
            flag_1_data.append(item)

    # 将 flag 为 0 的数据写入输出文件
    with open(output_file_flag_0, 'a') as f:
        json.dump(flag_0_data, f, indent=4)

    # 将 flag 为 1 的数据写入输出文件
    with open(output_file_flag_1, 'a') as f:
        json.dump(flag_1_data, f, indent=4)

# 使用示例
input_file = 'embeddings-challenge.json'  # 输入的 JSON 文件
output_file_flag_0 = 'flag_0_data.json'  # 存储 flag 为 0 的数据
output_file_flag_1 = 'flag_1_data.json'  # 存储 flag 为 1 的数据

# 调用函数进行分离和存储
separate_data_by_flag(input_file, output_file_flag_0, output_file_flag_1)


In [7]:
import json
import random

def extract_data_and_merge(input_file, output_file, percentage=0.2):
    # 读取输入的 JSON 文件
    with open(input_file, 'r') as f:
        data = json.load(f)

    # 分离 flag 为 0 和 flag 为 1 的数据
    flag_0_data = [item for item in data if item.get('flag') == 0]
    flag_1_data = [item for item in data if item.get('flag') == 1]

    # 计算需要从 flag 0 数据中抽取的条目数量
    num_to_extract = int(len(flag_1_data) * percentage)

    # 从 flag 0 数据中随机抽取 num_to_extract 条目
    extracted_data = random.sample(flag_0_data, num_to_extract)

    # 合并抽取的 flag 0 数据和所有 flag 1 的数据
    merged_data = extracted_data + flag_1_data

    # 随机打乱合并后的数据
    random.shuffle(merged_data)

    # 将打乱后的数据写入新的输出文件
    with open(output_file, 'w') as f:
        json.dump(merged_data, f, indent=4)

# 使用示例
input_file = 'embeddings-challenge.json'  # 输入的 JSON 文件
output_file = 'shuffled_merged_data2.json'  # 输出合成后的随机打乱 JSON 文件

# 调用函数抽取、合成和打乱数据
extract_data_and_merge(input_file, output_file, percentage=0.2)


In [8]:
import json

def merge_json_files(flag_0_file, flag_1_file, output_file):
    # 读取 flag_0_data.json 文件
    with open(flag_0_file, 'r') as f:
        flag_0_data = json.load(f)

    # 读取 flag_1_data.json 文件
    with open(flag_1_file, 'r') as f:
        flag_1_data = json.load(f)

    # 合并两个数据列表
    merged_data = flag_0_data + flag_1_data

    # 将合并后的数据写入新的 JSON 文件
    with open(output_file, 'w') as f:
        json.dump(merged_data, f, indent=4)

# 使用示例
flag_0_file = 'shuffled_merged_data.json'  # 输入的 flag=0 的 JSON 文件
flag_1_file = 'shuffled_merged_data2.json'  # 输入的 flag=1 的 JSON 文件
output_file = 'flag_1_81_flag_0_17.json'   # 输出合并后的 JSON 文件

# 调用函数合并文件
merge_json_files(flag_0_file, flag_1_file, output_file)
